# 💿 Dataset Conversion (Sensor-Timestamp Synchronized)

This notebook converts raw robot recordings (`.mcap` files) into the LeRobot format using **sensor-timestamp synchronization**.

## Key Differences from Standard Conversion

| Feature | Standard (`01_create_dataset.ipynb`) | Synced (this notebook) |
|---------|--------------------------------------|------------------------|
| Sync method | Log-time (arrival order) | Sensor timestamps |
| Frame generation | Based on message arrival | Synthetic timestamps at fixed FPS |
| Slow topics | May cause gaps | Reuses messages within tolerance |

**When to use this notebook:**
- Your sensors have varying frequencies
- You need precise temporal alignment across topics
- Standard conversion produces frame misalignment

The process involves:
1. **Exploring** the available raw data.
2. **Configuring** the dataset parameters (including sync tolerance).
3. **Running** the synchronized conversion.

--- 
## 1. Explore Raw Data

First, let's list the available raw data directories. Each directory contains a set of `.mcap` files from different teleoperation sessions.

In [ ]:
!du -sh /data/raw/*

--- 
## 2. Configure Conversion

Now, specify the input and output paths and define the dataset's structure.

### Sync-Specific Parameters

- **`TOLERANCE_MS`**: Maximum time difference (in milliseconds) allowed when matching messages. If `None`, it is auto-computed as `1000 / target_fps` (i.e., one frame period).

> **Action Required:** Update `RAW_DATA_DIR`, `OUTPUT_DIR`, and optionally `TOLERANCE_MS` below.

In [ ]:
import pathlib
from example_policies.data_ops.config.pipeline_config import PipelineConfig
from example_policies.data_ops.filters import FilterConfig
from example_policies.utils.action_order import ActionMode

# --- Paths ---
# TODO: Set the input directory containing your .mcap files.
RAW_DATA_DIR = pathlib.Path("/data/raw/your_dataset_here")

# TODO: Set your desired output directory name.
OUTPUT_DIR = pathlib.Path("/data/lerobot/your_dataset_here")

# --- Sync-Specific Configuration ---
# Maximum time difference for matching messages across topics.
# Set to None for auto-compute (1 frame period at target_fps).
TOLERANCE_MS = None  # e.g., 40.0 for 40ms tolerance

# Causal sync: only use past messages (no future lookahead).
# Set to False to allow matching future messages as well (nearest-neighbor).
CAUSAL = True

# Maximum number of episodes to process (None = no limit)
MAX_EPISODES = None  # e.g., 5 to process only first 5 episodes

# --- Episode Filtering ---
# Filter episodes based on metadata quality ratings
SUCCESS_ONLY = True   # Include only successful episodes (based on MCAP metadata)
EXCELLENT_ONLY = True  # Include only episodes with 'excellent' quality

# --- Platform API Filter (Optional) ---
# URL query string copied from the platform UI. When set, local filtering
# flags (SUCCESS_ONLY, EXCELLENT_ONLY, COMPLETE_SUBTASKS_ONLY) are ignored
# and episode selection is delegated to the platform API.
API_FILTER = None  # e.g., "task=pick_red_brick&rating=excellent&hide_ignored=true"

# --- Annotation Extraction (Optional) ---
# Extract segment annotations from MCAP metadata for use with LeRobot-Annotate
COMPLETE_SUBTASKS_ONLY = False  # Include only episodes where all defined subtasks have a successful segment
WITH_ANNOTATIONS = False  # Set to True to extract segment annotations

# --- Episode Quality Filters (Optional) ---
# When ENABLE_FILTERS=True, the new filter pipeline replaces the legacy
# FrameTargeter-based pause handling with three composable filters.
# Set to None to use the legacy behaviour (drop all paused frames).
ENABLE_FILTERS = True

filter_config = FilterConfig(
    # Pause filter — trims leading pauses (idle at start), keeps mid-episode
    # pauses but downgrades episode quality to "ok".
    enable_pause_filter=True,
    max_pause_seconds=0.2,       # Seconds of inactivity before classifying as pause
    pause_velocity=0.03,         # Joint velocity norm below which arm is "idle"
    trim_leading_pauses=True,    # Cut idle frames at the start of episodes
    trim_trailing_pauses=False,  # Optionally cut idle frames at the end

    # Gripper toggle filter — flags episodes where gripper toggles too fast.
    enable_gripper_toggle_filter=True,
    gripper_command_threshold=0.5,   # Value above which gripper is "closed"
    full_cycle_threshold_s=1.3,      # Max time for off→on→off cycle
    min_change_interval_s=0.65,      # Min time between consecutive changes

    # Gripper-while-moving filter — flags episodes where gripper command
    # changes while the arm is still in motion.
    enable_gripper_while_moving_filter=True,
    moving_velocity_threshold=0.03,  # Joint velocity norm for "arm is moving"
) if ENABLE_FILTERS else None

# --- Configuration ---
# TODO: A descriptive label for the task, used for VLA-style text conditioning.
TASK_LABEL = "Your Task Name Here"

cfg = PipelineConfig(
    task_name=TASK_LABEL,
    # Observation features to include in the dataset.
    include_tcp_poses=True,
    include_rgb_images=True,
    include_depth_images=False,
    include_last_command=False,
    # --- Action representation ---
    # ActionMode.TCP       → 16-dim absolute TCP (recommended default)
    # ActionMode.DELTA_TCP → 14-dim frame-to-frame delta
    #
    # For UMI-delta actions, keep ActionMode.TCP here and enable
    # use_chunk_relative_actions=True in the TRAINING config (notebook 02).
    # This converts abs TCP → 20-dim chunk-relative UMI-delta on-the-fly
    # during training, which is preferred over baking it into the dataset.
    action_level=ActionMode.TCP,
    # --- Gripper representation ---
    # True  → 1 value per side (summed finger joints = gripper width), 16-dim state
    # False → 2 finger-joint positions per side (legacy), 18-dim state
    use_single_gripper_value=False,
    # Subsampling and filtering. These are task-dependent.
    #recording_fps=20,
    target_fps=30,
    max_pause_seconds=0.2,
    min_episode_seconds=1,
)

# Compute actual tolerance
actual_tolerance_ms = TOLERANCE_MS if TOLERANCE_MS is not None else (1000.0 / cfg.target_fps)

print(f"Input path:  {RAW_DATA_DIR}")
print(f"Output path: {OUTPUT_DIR}")
print(f"\nSync Configuration:")
print(f"  - Target FPS: {cfg.target_fps} Hz")
print(f"  - Tolerance: {actual_tolerance_ms:.1f} ms {'(auto)' if TOLERANCE_MS is None else ''}")
print(f"  - Causal (past-only): {CAUSAL}")
print(f"  - Action level: {cfg.action_level}")
print(f"  - Single gripper value: {cfg.use_single_gripper_value}")
print(f"  - Max episodes: {MAX_EPISODES if MAX_EPISODES else 'All'}")
print(f"\nEpisode Filtering:")
if API_FILTER:
    print(f"  - API filter: {API_FILTER}")
else:
    print(f"  - Success only: {SUCCESS_ONLY}")
    print(f"  - Excellent only: {EXCELLENT_ONLY}")
    print(f"  - Complete subtasks only: {COMPLETE_SUBTASKS_ONLY}")
print(f"\nAnnotations:")
print(f"  - Extract annotations: {WITH_ANNOTATIONS}")
print(f"\nQuality Filters:")
if filter_config:
    print(f"  - Pause filter: {filter_config.enable_pause_filter} (trim leading: {filter_config.trim_leading_pauses})")
    print(f"  - Gripper toggle filter: {filter_config.enable_gripper_toggle_filter}")
    if filter_config.enable_gripper_toggle_filter:
        print(f"    - Full cycle threshold: {filter_config.full_cycle_threshold_s}s")
        print(f"    - Min change interval: {filter_config.min_change_interval_s}s")
    print(f"  - Gripper-while-moving filter: {filter_config.enable_gripper_while_moving_filter}")
    print(f"  - Min quality to include: {filter_config.min_quality}")
else:
    print(f"  - Disabled (using legacy pause handling)")

--- 
## 2.5 Debug: Analyze Topic Frequencies

Before running the conversion, let's analyze the raw MCAP files to see what topics are available and at what frequencies they're recorded. This is especially important for synced conversion to ensure your tolerance setting is appropriate.

In [ ]:
from mcap.reader import make_reader
from rosbags.typesys import Stores, get_typestore
from collections import defaultdict
import numpy as np
import sys
from example_policies.data_ops.config.rosbag_topics import RosTopicEnum
from example_policies.data_ops.utils.conversion_utils import get_selected_episodes

# Initialize typestore for ROS2 message deserialization
typestore = get_typestore(Stores.ROS2_HUMBLE)

def extract_sensor_timestamp(msg) -> float | None:
    """Extract sensor timestamp from a decoded ROS message header.
    
    Returns timestamp in nanoseconds, or None if no header available.
    """
    # Most ROS messages with timestamps have a 'header' field
    if hasattr(msg, "header") and hasattr(msg.header, "stamp"):
        stamp = msg.header.stamp
        return stamp.sec * 1e9 + stamp.nanosec
    
    # Some messages have a direct 'stamp' field
    if hasattr(msg, "stamp"):
        stamp = msg.stamp
        return stamp.sec * 1e9 + stamp.nanosec
    
    return None

def analyze_mcap_topics(mcap_path, use_sensor_time=True):
    """Analyze topics and their frequencies in an MCAP file.
    
    Args:
        mcap_path: Path to MCAP file
        use_sensor_time: If True, use sensor timestamp from header when available,
                         falling back to log_time. If False, always use log_time.
    """
    topic_stats = defaultdict(lambda: {"count": 0, "timestamps": [], "uses_sensor_time": False})
    # Map channel topic to schema name for deserialization
    topic_to_schema = {}
    
    with open(mcap_path, 'rb') as f:
        reader = make_reader(f)
        for schema, channel, message in reader.iter_messages():
            topic = channel.topic
            
            # Store schema name for this topic
            if topic not in topic_to_schema and schema is not None:
                topic_to_schema[topic] = schema.name
            
            # Try to get sensor timestamp from message header
            sensor_ts = None
            if use_sensor_time and topic in topic_to_schema:
                try:
                    decoded_msg = typestore.deserialize_cdr(message.data, topic_to_schema[topic])
                    sensor_ts = extract_sensor_timestamp(decoded_msg)
                except Exception:
                    pass  # Fall back to log_time if deserialization fails
            
            if sensor_ts is not None:
                timestamp = sensor_ts
                topic_stats[topic]["uses_sensor_time"] = True
            else:
                timestamp = message.log_time  # nanoseconds
            
            topic_stats[topic]["count"] += 1
            topic_stats[topic]["timestamps"].append(timestamp)
    
    # Calculate frequencies
    results = {}
    for topic, stats in topic_stats.items():
        if len(stats["timestamps"]) > 1:
            timestamps_sec = np.array(stats["timestamps"]) / 1e9  # convert to seconds
            time_diffs = np.diff(timestamps_sec)
            
            # Count invalid time diffs
            invalid_count = np.sum(time_diffs <= 0)
            
            # Filter out zero or negative time diffs
            valid_diffs = time_diffs[time_diffs > 0]
            
            if len(valid_diffs) > 0:
                avg_freq = 1.0 / np.mean(valid_diffs)
                min_freq = 1.0 / np.max(valid_diffs)  # Longest gap = lowest frequency
                max_freq = 1.0 / np.min(valid_diffs)  # Shortest gap = highest frequency
                std_freq = np.std(1.0 / valid_diffs)  # Std of instantaneous frequencies
            else:
                avg_freq = 0
                min_freq = 0
                max_freq = 0
                std_freq = 0
            
            results[topic] = {
                "message_count": stats["count"],
                "avg_frequency_hz": avg_freq,
                "min_frequency_hz": min_freq,
                "max_frequency_hz": max_freq,
                "std_frequency_hz": std_freq,
                "duration_sec": timestamps_sec[-1] - timestamps_sec[0],
                "invalid_time_diffs": int(invalid_count),
                "uses_sensor_time": stats["uses_sensor_time"],
            }
        else:
            results[topic] = {
                "message_count": stats["count"],
                "avg_frequency_hz": 0,
                "min_frequency_hz": 0,
                "max_frequency_hz": 0,
                "std_frequency_hz": 0,
                "duration_sec": 0,
                "invalid_time_diffs": 0,
                "uses_sensor_time": stats["uses_sensor_time"],
            }
    
    return results

# Define topics of interest with new and old names
TOPICS_OF_INTEREST = [
    ("RGB Left Image", [RosTopicEnum.RGB_LEFT_IMAGE.value]),
    ("RGB Right Image", [RosTopicEnum.RGB_RIGHT_IMAGE.value]),
    ("RGB Static Image", [RosTopicEnum.RGB_STATIC_IMAGE.value]),
    ("Desired Gripper Left", [RosTopicEnum.DES_GRIPPER_LEFT.value]),
    ("Desired Gripper Right", [RosTopicEnum.DES_GRIPPER_RIGHT.value]),
    ("Desired TCP Left", ["/cartesian_target_left", "/desired_pose_twist_left"]),
    ("Desired TCP Right", ["/cartesian_target_right", "/desired_pose_twist_right"]),
    ("Actual Joint State", [RosTopicEnum.ACTUAL_JOINT_STATE.value]),
    ("Actual TCP Left", [RosTopicEnum.ACTUAL_TCP_LEFT.value]),
    ("Actual TCP Right", [RosTopicEnum.ACTUAL_TCP_RIGHT.value]),
]

# Get valid episodes using the standard function
episode_paths = get_selected_episodes(RAW_DATA_DIR, success_only=True)

# TODO: Change this index to analyze a different episode
SELECTED_EPISODE_INDEX = 0

# Build output as a single string to avoid duplicate printing
output = []
output.append(f"Found {len(episode_paths)} valid episodes\n")

if episode_paths:
    # Validate index
    if SELECTED_EPISODE_INDEX >= len(episode_paths):
        SELECTED_EPISODE_INDEX = 0
        output.append(f"⚠️  Index out of range, using episode 0\n")
    
    selected_episode_path = episode_paths[SELECTED_EPISODE_INDEX]
    output.append(f"Analyzing MCAP file [{SELECTED_EPISODE_INDEX}]: {selected_episode_path.name}\n")
    output.append("="*80 + "\n")
    
    topic_stats = analyze_mcap_topics(selected_episode_path, use_sensor_time=True)
    
    # Check if tolerance is appropriate
    output.append(f"\n🔧 Sync Tolerance Check (configured: {actual_tolerance_ms:.1f} ms):\n")
    output.append("-"*80 + "\n")
    
    for topic_label, topic_names in TOPICS_OF_INTEREST:
        found_topic = None
        for topic_name in topic_names:
            if topic_name in topic_stats:
                found_topic = topic_name
                break
        
        if found_topic:
            stats = topic_stats[found_topic]
            avg_period_ms = 1000.0 / stats['avg_frequency_hz'] if stats['avg_frequency_hz'] > 0 else float('inf')
            
            # Check if tolerance is appropriate for this topic
            if avg_period_ms > actual_tolerance_ms * 2:
                status = "⚠️  SLOW (may miss frames)"
            elif avg_period_ms > actual_tolerance_ms:
                status = "⚡ OK (some reuse)"
            else:
                status = "✅ FAST"
            
            # Indicate timestamp source
            ts_source = "🕐 sensor" if stats['uses_sensor_time'] else "📝 log"
            
            output.append(f"  {topic_label}: {stats['avg_frequency_hz']:.1f} Hz (period: {avg_period_ms:.1f} ms) - {status} [{ts_source}]\n")
    
    output.append("\n" + "="*80 + "\n")
    
    # Print all topics found
    output.append("\n📋 All topics in MCAP file:\n")
    output.append("-"*80 + "\n")
    for topic, stats in sorted(topic_stats.items()):
        ts_source = "sensor" if stats['uses_sensor_time'] else "log"
        output.append(f"  {topic}: {stats['message_count']} msgs, {stats['avg_frequency_hz']:.2f} Hz [{ts_source}]\n")
else:
    output.append(f"No valid episodes found in {RAW_DATA_DIR}\n")
    output.append(f"💡 Try running with success_only=False to see all files\n")

# Print all at once
sys.stdout.write(''.join(output))
sys.stdout.flush()

--- 
## 2.6 Visualize Topic Time Step Distribution

Visualize the distribution of time steps (time differences between consecutive messages) for a specific topic. This helps determine if your tolerance setting is appropriate.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from mcap.reader import make_reader
from collections import defaultdict
from datetime import datetime
import pathlib

# Configuration
FIX_X_AXIS_RANGE = False  # Set to False to show full data range
X_AXIS_RANGE_PERCENT = 50  # Show mean ± this percentage

# PDF output path - save to notebooks/outputs directory (writable)
PDF_OUTPUT_DIR = pathlib.Path("outputs")
PDF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PDF_OUTPUT_PATH = PDF_OUTPUT_DIR / f"topic_analysis_{RAW_DATA_DIR.name}.pdf"

# Create plots for all topics of interest
if episode_paths:
    # Use the same episode selected in the previous cell
    print(f"Visualizing episode [{SELECTED_EPISODE_INDEX}]: {selected_episode_path.name}")
    print(f"Configured tolerance: {actual_tolerance_ms:.1f} ms (shown as vertical red line)")
    print(f"Using sensor timestamps when available (fallback to log_time)\n")
    
    with PdfPages(PDF_OUTPUT_PATH) as pdf:
        # --- First page: Sync Tolerance Check Summary ---
        fig_summary = plt.figure(figsize=(12, 10))
        ax_summary = fig_summary.add_subplot(111)
        ax_summary.axis('off')
        
        # Build the sync tolerance check text (use ASCII for PDF compatibility)
        summary_lines = []
        summary_lines.append(f"[WRENCH] Sync Tolerance Check Report")
        summary_lines.append(f"{'='*60}")
        summary_lines.append(f"")
        summary_lines.append(f"Episode: {selected_episode_path.name}")
        summary_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        summary_lines.append(f"Configured Tolerance: {actual_tolerance_ms:.1f} ms")
        summary_lines.append(f"Target FPS: {cfg.target_fps} Hz")
        summary_lines.append(f"")
        summary_lines.append(f"{'-'*60}")
        summary_lines.append(f"")
        
        for topic_label, topic_names in TOPICS_OF_INTEREST:
            found_topic = None
            for topic_name in topic_names:
                if topic_name in topic_stats:
                    found_topic = topic_name
                    break
            
            if found_topic:
                stats = topic_stats[found_topic]
                avg_period_ms = 1000.0 / stats['avg_frequency_hz'] if stats['avg_frequency_hz'] > 0 else float('inf')
                
                # Check if tolerance is appropriate for this topic (ASCII symbols for PDF)
                if avg_period_ms > actual_tolerance_ms * 2:
                    status = "[!!] SLOW (may miss frames)"
                elif avg_period_ms > actual_tolerance_ms:
                    status = "[~] OK (some reuse)"
                else:
                    status = "[OK] FAST"
                
                ts_source = "[sensor]" if stats['uses_sensor_time'] else "[log]"
                summary_lines.append(f"{topic_label}:")
                summary_lines.append(f"    {stats['avg_frequency_hz']:.1f} Hz (period: {avg_period_ms:.1f} ms)")
                summary_lines.append(f"    {status} {ts_source}")
                summary_lines.append(f"")
        
        summary_text = '\n'.join(summary_lines)
        ax_summary.text(0.05, 0.95, summary_text, transform=ax_summary.transAxes,
                       verticalalignment='top', horizontalalignment='left',
                       fontsize=11, fontfamily='monospace',
                       bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.3))
        
        fig_summary.suptitle('Topic Frequency Analysis Summary', fontsize=16, fontweight='bold')
        pdf.savefig(fig_summary, bbox_inches='tight')
        plt.close(fig_summary)
        
        # --- Subsequent pages: Distribution plots for each topic ---
        for topic_label, topic_names in TOPICS_OF_INTEREST:
            # Find which version of the topic exists and collect timestamps
            found_topic = None
            timestamps = []
            uses_sensor_time = False
            
            for topic_name in topic_names:
                timestamps = []
                uses_sensor_time = False
                schema_name = None
                
                with open(selected_episode_path, 'rb') as f:
                    reader = make_reader(f)
                    for schema, channel, message in reader.iter_messages():
                        if channel.topic == topic_name:
                            # Get schema name for deserialization
                            if schema_name is None and schema is not None:
                                schema_name = schema.name
                            
                            # Try to get sensor timestamp
                            sensor_ts = None
                            if schema_name is not None:
                                try:
                                    decoded_msg = typestore.deserialize_cdr(message.data, schema_name)
                                    sensor_ts = extract_sensor_timestamp(decoded_msg)
                                except Exception:
                                    pass
                            
                            if sensor_ts is not None:
                                timestamps.append(sensor_ts / 1e9)  # Convert to seconds
                                uses_sensor_time = True
                            else:
                                timestamps.append(message.log_time / 1e9)  # Convert to seconds
                
                if len(timestamps) > 0:
                    found_topic = topic_name
                    break
            
            if not found_topic or len(timestamps) <= 1:
                continue
            
            # Calculate time differences (in milliseconds for better readability)
            time_diffs_ms = np.diff(timestamps) * 1000
            
            # Filter out invalid diffs for visualization
            valid_diffs = time_diffs_ms[time_diffs_ms > 0]
            
            if len(valid_diffs) > 0:
                # Create the distribution plot
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
                
                # Determine timestamp source label
                ts_label = "sensor time" if uses_sensor_time else "log time"
                
                # Histogram
                ax1.hist(valid_diffs, bins=50, edgecolor='black', alpha=0.7)
                ax1.axvline(x=actual_tolerance_ms, color='red', linestyle='--', linewidth=2, label=f'Tolerance ({actual_tolerance_ms:.1f} ms)')
                ax1.set_xlabel('Time Step (ms)', fontsize=12)
                ax1.set_ylabel('Count', fontsize=12)
                ax1.set_title(f'Distribution of Time Steps ({ts_label})\n{topic_label} ({found_topic})', fontsize=14)
                ax1.grid(True, alpha=0.3)
                ax1.legend()
                
                # Optionally fix x-axis range to mean ± percentage
                mean_val = np.mean(valid_diffs)
                if FIX_X_AXIS_RANGE:
                    x_min = mean_val * (1 - X_AXIS_RANGE_PERCENT / 100)
                    x_max = mean_val * (1 + X_AXIS_RANGE_PERCENT / 100)
                    ax1.set_xlim(x_min, x_max)
                
                # Add statistics text
                stats_text = f'Mean: {np.mean(valid_diffs):.2f} ms\n'
                stats_text += f'Median: {np.median(valid_diffs):.2f} ms\n'
                stats_text += f'Std: {np.std(valid_diffs):.2f} ms\n'
                stats_text += f'Min: {np.min(valid_diffs):.2f} ms\n'
                stats_text += f'Max: {np.max(valid_diffs):.2f} ms\n'
                stats_text += f'Freq: {1000/np.mean(valid_diffs):.2f} Hz'
                ax1.text(0.98, 0.98, stats_text, transform=ax1.transAxes,
                        verticalalignment='top', horizontalalignment='right',
                        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                        fontsize=10)
                
                # Box plot for outlier detection (horizontal)
                bp = ax2.boxplot(valid_diffs, vert=False)
                ax2.axvline(x=actual_tolerance_ms, color='red', linestyle='--', linewidth=2, label=f'Tolerance ({actual_tolerance_ms:.1f} ms)')
                ax2.set_xlabel('Time Step (ms)', fontsize=12)
                ax2.set_title('Box Plot (Outlier Detection)', fontsize=14)
                ax2.grid(True, alpha=0.3, axis='x')
                ax2.legend()
                
                # Match x-axis range with histogram if fixed
                if FIX_X_AXIS_RANGE:
                    ax2.set_xlim(x_min, x_max)
                
                plt.tight_layout()
                
                # Save to PDF and display
                pdf.savefig(fig, bbox_inches='tight')
                plt.show()
                
                ts_indicator = "🕐 sensor time" if uses_sensor_time else "📝 log time"
                print(f"Analyzed {len(timestamps)} messages for {topic_label} [{ts_indicator}]")
                print(f"Valid time steps: {len(valid_diffs)}, Invalid: {len(time_diffs_ms) - len(valid_diffs)}\n")
    
    print(f"\n📄 All visualizations saved to: {PDF_OUTPUT_PATH}")
else:
    print(f"No MCAP files found in {RAW_DATA_DIR}")

--- 
## 3. Run Synchronized Conversion

This cell executes the **sensor-timestamp synchronized** conversion process. 

### How it works:
1. **Ingest all messages** from the MCAP file with their sensor timestamps
2. **Generate synthetic timestamps** at a fixed output frequency (`target_fps`)
3. **Find nearest messages** from all topics within the tolerance window
4. **Assemble frames** and write to LeRobot format

This may take a while depending on the size of your data.

In [ ]:
import shutil
from example_policies.data_ops.dataset_conversion_synced import (
    convert_episodes_synced,
    print_conversion_result,
)
from example_policies.data_ops.utils.conversion_utils import get_selected_episodes

# --- Option to delete existing dataset ---
DELETE_EXISTING_DATASET = True  # Set to True to delete existing dataset before conversion

if DELETE_EXISTING_DATASET and OUTPUT_DIR.exists():
    print(f"🗑️  Deleting existing dataset at {OUTPUT_DIR}...")
    shutil.rmtree(OUTPUT_DIR)
    print("   Done.\n")

# Get episodes with filtering
episode_paths = get_selected_episodes(
    RAW_DATA_DIR, 
    success_only=SUCCESS_ONLY,
    excellent_only=EXCELLENT_ONLY,
    complete_subtasks_only=COMPLETE_SUBTASKS_ONLY,
    api_filter=API_FILTER,
)

# Optionally limit number of episodes
if MAX_EPISODES is not None:
    episode_paths = episode_paths[:MAX_EPISODES]
    print(f"Limiting to first {MAX_EPISODES} episodes")

print(f"Converting {len(episode_paths)} episodes...")
print(f"Sync settings: target_fps={cfg.target_fps}, tolerance={actual_tolerance_ms:.1f}ms, causal={CAUSAL}")
if filter_config:
    print("Quality filters: ENABLED")
if WITH_ANNOTATIONS:
    print("Annotation extraction: ENABLED")
print()

# Run synchronized conversion
result = convert_episodes_synced(
    episode_paths=episode_paths,
    output_dir=OUTPUT_DIR,
    config=cfg,
    tolerance_ms=actual_tolerance_ms,
    with_annotations=WITH_ANNOTATIONS,
    causal=CAUSAL,
    filter_config=filter_config,
)

# Save annotations if extracted
if WITH_ANNOTATIONS and result.get("annotation_extractor"):
    extractor = result["annotation_extractor"]
    if extractor.total_segments > 0:
        print("\n📝 Saving annotations...")
        extractor.save(OUTPUT_DIR)
        print(f"  - Episodes with annotations: {extractor.episodes_with_annotations}")
        print(f"  - Total segments: {extractor.total_segments}")
    else:
        print("\n⚠️  No segment annotations found in episodes.")

# Print results
print_conversion_result(result)

---
## 4. Verify Output

Let's verify the converted dataset was created successfully.

In [ ]:
import json

# Check output directory contents
print(f"Output directory: {OUTPUT_DIR}")
print("\nContents:")
!ls -la {OUTPUT_DIR}

# Show metadata if available
metadata_path = OUTPUT_DIR / "meta" / "info.json"
if metadata_path.exists():
    print("\n" + "="*50)
    print("Dataset Info:")
    print("="*50)
    with open(metadata_path) as f:
        info = json.load(f)
    print(f"  FPS: {info.get('fps', 'N/A')}")
    print(f"  Total episodes: {info.get('total_episodes', 'N/A')}")
    print(f"  Total frames: {info.get('total_frames', 'N/A')}")
    print(f"  Features: {list(info.get('features', {}).keys())}")

### 4.1 Verify Annotations (Optional)

If you enabled `WITH_ANNOTATIONS=True`, the dataset will include annotation files in the `meta/` directory. These are compatible with [LeRobot-Annotate](https://github.com/huggingface/lerobot-annotate) for visualization and further annotation.

In [ ]:
import json

# Check for annotation files
annotations_path = OUTPUT_DIR / "meta" / "lerobot_annotations.json"
subtasks_path = OUTPUT_DIR / "meta" / "subtasks.parquet"

if annotations_path.exists():
    print("✅ Annotations found!")
    print(f"\nAnnotation files:")
    print(f"  - {annotations_path}")
    if subtasks_path.exists():
        print(f"  - {subtasks_path}")
    
    # Show annotation summary
    with open(annotations_path) as f:
        annotations = json.load(f)
    
    episodes = annotations.get("episodes", {})
    total_segments = sum(len(ep.get("subtasks", [])) for ep in episodes.values())
    
    print(f"\nAnnotation Summary:")
    print(f"  - Annotated episodes: {len(episodes)}")
    print(f"  - Total segments: {total_segments}")
    
    # Show sample of first episode's subtasks
    if episodes:
        first_ep_id = list(episodes.keys())[0]
        first_ep = episodes[first_ep_id]
        subtasks = first_ep.get("subtasks", [])[:10]  # First 10 subtasks
        if subtasks:
            print(f"\nSample subtasks from episode {first_ep_id}:")
            for st in subtasks:
                print(f"    {st['start']:.2f}s - {st['end']:.2f}s: {st['label']}")
            if len(first_ep.get("subtasks", [])) > 10:
                print(f"    ... and {len(first_ep.get('subtasks', [])) - 10} more")
else:
    print("ℹ️  No annotations found (WITH_ANNOTATIONS was likely False)")
    print("   Set WITH_ANNOTATIONS=True in the configuration cell to extract annotations.")

---
## 5. Upload Dataset to Hugging Face

Upload your converted dataset to Hugging Face Hub for sharing and easy access during training.

> **Prerequisites:** 
> - You need a Hugging Face account and access token
> - Run `huggingface-cli login` in a terminal first, or set the `HF_TOKEN` environment variable

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from huggingface_hub import HfApi, login

# --- Hugging Face Upload Configuration ---
# TODO: Set your Hugging Face username/organization and dataset name
HF_REPO_ID = "pokeandwiggle/datasetname"  # e.g., "my-org/stack-blocks-synced"

# Whether to make the dataset private (requires HF Pro or organization)
PRIVATE = True

# Use upload_large_folder for datasets with many/large files (recommended for video datasets)
USE_LARGE_FOLDER_UPLOAD = True

# --- Ensure we're logged in ---
# Uncomment the line below if you need to log in interactively
# login()

# --- Create repo if it doesn't exist ---
api = HfApi()
try:
    api.repo_info(repo_id=HF_REPO_ID, repo_type="dataset")
    print(f"✓ Repository {HF_REPO_ID} already exists")
except Exception:
    print(f"Creating new dataset repository: {HF_REPO_ID}")
    api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", private=PRIVATE)
    print(f"✓ Created repository: {HF_REPO_ID}")

# --- Load and upload the dataset ---
print(f"\nLoading dataset from {OUTPUT_DIR}...")
dataset = LeRobotDataset(repo_id=HF_REPO_ID, root=OUTPUT_DIR)

print(f"Uploading to Hugging Face Hub: {HF_REPO_ID}")
print(f"  Private: {PRIVATE}")
print(f"  Large folder upload: {USE_LARGE_FOLDER_UPLOAD}")
print(f"  Episodes: {dataset.meta.total_episodes}")
print(f"  Frames: {dataset.meta.total_frames}")
print()

dataset.push_to_hub(
    repo_id=HF_REPO_ID,
    private=PRIVATE,
    upload_large_folder=USE_LARGE_FOLDER_UPLOAD,
)

print(f"\n✅ Dataset uploaded successfully!")
print(f"   View at: https://huggingface.co/datasets/{HF_REPO_ID}")

--- 
## ✅ Done!

Your new **synchronized** dataset is ready at the output path you specified. 

**Next steps:**
- Proceed to `02_train_model_simple.ipynb` to train a policy
- Use `04_upload_and_visualize_datasets.ipynb` to visualize the dataset
- If you extracted annotations, use [LeRobot-Annotate](https://github.com/huggingface/lerobot-annotate) to view/edit them

**Troubleshooting:**
- If you see many "skipped pauses", the robot may have been idle for extended periods
- If frames are missing, try increasing `TOLERANCE_MS`
- If frames are duplicated/stuttering, try decreasing `TOLERANCE_MS`
- If annotations are missing, ensure your MCAP files have segment metadata (`pw_episode_info` in new format, or `segments` + `episode_rating` in old format)